In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [2]:
# ══════════════════════════════════════════════
#  CHARGEMENT & PRÉPARATION
# ══════════════════════════════════════════════
df = pd.read_csv('career_cleaned_ml.csv')

# ─── Regrouper les métiers en 8 catégories ────
def group_job(title):
    t = str(title).lower()
    if any(x in t for x in ['software','developer','programmer','web','devops','asp','sql developer','front','back']):
        return 'Software / Dev'
    elif any(x in t for x in ['data','analyst','scientist','machine learning','ai']):
        return 'Data / Analytics'
    elif any(x in t for x in ['mechanical','civil','structural','production','design engineer','site','quality']):
        return 'Mechanical / Civil'
    elif any(x in t for x in ['computer software','system','network','cyber','it','instrumentation']):
        return 'IT / Systems'
    elif any(x in t for x in ['business','consultant','project manager','associate consultant']):
        return 'Business / Consulting'
    elif any(x in t for x in ['teacher','professor','research']):
        return 'Education / Research'
    elif any(x in t for x in ['sales','marketing','hr','executive','tele','financial']):
        return 'Sales / HR / Finance'
    else:
        return 'Other'

df['job_category'] = df['job_title'].apply(group_job)

# ─── Encodage des features ────────────────────
df_enc = df.copy()

for col in ['gender', 'ug_degree', 'has_certification', 'is_working']:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

def group_spec(s):
    s = str(s).lower()
    if any(x in s for x in ['computer','software','it','data','engineering','electronic','information']):
        return 0
    elif any(x in s for x in ['commerce','finance','business','accounting','economics','management']):
        return 1
    elif any(x in s for x in ['psychology','arts','literature','history','sociology']):
        return 2
    elif any(x in s for x in ['maths','mathematics','physics','chemistry','science']):
        return 3
    else:
        return 4

df_enc['spec_group']   = df_enc['ug_specialization'].apply(group_spec)
df_enc['skills_count'] = df_enc['skills'].astype(str).str.split(r'[;,]').apply(len)
df_enc['has_masters']  = (df_enc['masters_field'].astype(str).str.lower() != 'none').astype(int)

feature_cols = ['gender', 'ug_degree', 'spec_group', 'cgpa_normalized',
                'has_certification', 'is_working', 'skills_count', 'has_masters']

X = df_enc[feature_cols]
y = df_enc['job_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
X_sc       = scaler.fit_transform(X)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Classes: {sorted(y.unique())}\n")

Train: (301, 8) | Test: (76, 8)
Classes: ['Business / Consulting', 'Data / Analytics', 'Education / Research', 'IT / Systems', 'Mechanical / Civil', 'Other', 'Sales / HR / Finance', 'Software / Dev']



In [3]:
# ══════════════════════════════════════════════
#  MODÈLE 1 — Decision Tree
# ══════════════════════════════════════════════
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
dt_test = accuracy_score(y_test, dt.predict(X_test))
dt_cv   = cross_val_score(dt, X, y, cv=5).mean()

print("=" * 50)
print("  MODÈLE 1 — Decision Tree")
print("=" * 50)
print(f"  Accuracy Test : {dt_test*100:.2f}%")
print(f"  Accuracy CV5  : {dt_cv*100:.2f}%")
print(classification_report(y_test, dt.predict(X_test)))

  MODÈLE 1 — Decision Tree
  Accuracy Test : 27.63%
  Accuracy CV5  : 27.87%
                       precision    recall  f1-score   support

Business / Consulting       0.00      0.00      0.00         3
     Data / Analytics       0.27      0.20      0.23        15
 Education / Research       0.25      0.12      0.17         8
         IT / Systems       0.00      0.00      0.00         1
   Mechanical / Civil       0.23      0.20      0.21        15
                Other       0.25      0.25      0.25         4
 Sales / HR / Finance       0.50      0.33      0.40         6
       Software / Dev       0.33      0.46      0.39        24

             accuracy                           0.28        76
            macro avg       0.23      0.20      0.21        76
         weighted avg       0.28      0.28      0.27        76



In [4]:
# ══════════════════════════════════════════════
#  MODÈLE 2 — Random Forest
# ══════════════════════════════════════════════
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_test = accuracy_score(y_test, rf.predict(X_test))
rf_cv   = cross_val_score(rf, X, y, cv=5).mean()

print("=" * 50)
print("  MODÈLE 2 — Random Forest")
print("=" * 50)
print(f"  Accuracy Test : {rf_test*100:.2f}%")
print(f"  Accuracy CV5  : {rf_cv*100:.2f}%")
print(classification_report(y_test, rf.predict(X_test)))


  MODÈLE 2 — Random Forest
  Accuracy Test : 23.68%
  Accuracy CV5  : 29.18%
                       precision    recall  f1-score   support

Business / Consulting       0.00      0.00      0.00         3
     Data / Analytics       0.11      0.13      0.12        15
 Education / Research       0.50      0.38      0.43         8
         IT / Systems       0.00      0.00      0.00         1
   Mechanical / Civil       0.26      0.40      0.32        15
                Other       0.00      0.00      0.00         4
 Sales / HR / Finance       0.00      0.00      0.00         6
       Software / Dev       0.32      0.29      0.30        24

             accuracy                           0.24        76
            macro avg       0.15      0.15      0.15        76
         weighted avg       0.23      0.24      0.23        76



In [5]:
# ══════════════════════════════════════════════
#  MODÈLE 3 — KNN  (best K auto)
# ══════════════════════════════════════════════
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
             X_sc, y, cv=5).mean() for k in range(1, 21)]
best_k = range(1, 21)[cv_scores.index(max(cv_scores))]

knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_sc, y_train)
knn_test = accuracy_score(y_test, knn.predict(X_test_sc))
knn_cv   = max(cv_scores)

print("=" * 50)
print(f"  MODÈLE 3 — KNN  (K={best_k})")
print("=" * 50)
print(f"  Accuracy Test : {knn_test*100:.2f}%")
print(f"  Accuracy CV5  : {knn_cv*100:.2f}%")
print(classification_report(y_test, knn.predict(X_test_sc)))

  MODÈLE 3 — KNN  (K=15)
  Accuracy Test : 23.68%
  Accuracy CV5  : 31.01%
                       precision    recall  f1-score   support

Business / Consulting       0.00      0.00      0.00         3
     Data / Analytics       0.10      0.13      0.11        15
 Education / Research       0.00      0.00      0.00         8
         IT / Systems       0.00      0.00      0.00         1
   Mechanical / Civil       0.27      0.27      0.27        15
                Other       0.00      0.00      0.00         4
 Sales / HR / Finance       0.00      0.00      0.00         6
       Software / Dev       0.33      0.50      0.40        24

             accuracy                           0.24        76
            macro avg       0.09      0.11      0.10        76
         weighted avg       0.18      0.24      0.20        76



In [9]:
# ══════════════════════════════════════════════
#  MODÈLE 4 — Logistic Regression
# ══════════════════════════════════════════════
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_test = accuracy_score(y_test, lr.predict(X_test_sc))
lr_cv   = cross_val_score(lr, X_sc, y, cv=5).mean()

print("=" * 50)
print("  MODÈLE 4 — Logistic Regression")
print("=" * 50)
print(f"  Accuracy Test : {lr_test*100:.2f}%")
print(f"  Accuracy CV5  : {lr_cv*100:.2f}%")
print(classification_report(y_test, lr.predict(X_test_sc)))

  MODÈLE 4 — Logistic Regression
  Accuracy Test : 25.00%
  Accuracy CV5  : 31.01%
                       precision    recall  f1-score   support

Business / Consulting       0.00      0.00      0.00         3
     Data / Analytics       0.15      0.13      0.14        15
 Education / Research       0.00      0.00      0.00         8
         IT / Systems       0.00      0.00      0.00         1
   Mechanical / Civil       0.24      0.27      0.25        15
                Other       0.00      0.00      0.00         4
 Sales / HR / Finance       0.00      0.00      0.00         6
       Software / Dev       0.32      0.54      0.40        24

             accuracy                           0.25        76
            macro avg       0.09      0.12      0.10        76
         weighted avg       0.18      0.25      0.20        76



In [11]:
# ══════════════════════════════════════════════
#  RÉSUMÉ FINAL
# ══════════════════════════════════════════════
print("\n" + "=" * 50)
print("  RÉSUMÉ FINAL")
print("=" * 50)
results = list(zip(models, test_ac, cv_ac))
results.sort(key=lambda x: x[2], reverse=True)
for name, test, cv in results:
    star = " ⭐" if name == results[0][0] else ""
    print(f"  {name:22s} | Test={test*100:.1f}%  CV={cv*100:.1f}%{star}")

print(f"\nGraphique sauvegarde : outputs/comparaison_modeles.png")


  RÉSUMÉ FINAL
  KNN (K=15)             | Test=23.7%  CV=31.0% ⭐
  Logistic Reg.          | Test=25.0%  CV=31.0%
  Random Forest          | Test=23.7%  CV=29.2%
  Decision Tree          | Test=27.6%  CV=27.9%

Graphique sauvegarde : outputs/comparaison_modeles.png
